In [1]:
!pip install fastapi uvicorn FlagEmbedding pyngrok torch transformers nest-asyncio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 126.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 99.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 13.5 MB/s eta 0:

In [2]:
import asyncio
import time
import json
import gc
from typing import List, Dict, Optional, Tuple, Any
from dataclasses import dataclass
from concurrent.futures import ThreadPoolExecutor
import threading
from queue import Queue, Empty
import uuid

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from FlagEmbedding import FlagReranker
import numpy as np

from fastapi import FastAPI, HTTPException, BackgroundTasks
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
import uvicorn
from pyngrok import ngrok

In [3]:
# Configuration
MAX_CONCURRENT_REQUESTS = 1000  # A100 can handle many concurrent requests
BATCH_SIZE = 512  # Optimal batch size for A100
MODEL_NAME = 'BAAI/bge-reranker-v2-m3'
MAX_SEQUENCE_LENGTH = 512
QUEUE_TIMEOUT = 30.0
PROCESSING_TIMEOUT = 15.0

In [4]:
# Request/Response Models
class RerankRequest(BaseModel):
    query: str = Field(..., description="Search query")
    passages: List[str] = Field(..., description="List of passages to rerank")
    top_k: int = Field(default=3, ge=1, le=100, description="Number of top results to return")
    request_id: Optional[str] = Field(default=None, description="Optional request ID for tracking")

class BatchRerankRequest(BaseModel):
    queries: List[str] = Field(..., description="List of queries")
    passages: List[str] = Field(..., description="List of passages (same for all queries)")
    top_k: int = Field(default=3, ge=1, le=100, description="Number of top results per query")
    request_id: Optional[str] = Field(default=None, description="Optional request ID for tracking")

class PassageResult(BaseModel):
    passage: str
    score: float
    original_index: int

class RerankResponse(BaseModel):
    query: str
    top_passages: List[PassageResult]
    timing: Dict[str, float]
    total_passages: int
    top_k: int
    request_id: Optional[str] = None

class BatchRerankResponse(BaseModel):
    results: List[RerankResponse]
    timing: Dict[str, float]
    request_id: Optional[str] = None

class Pair(BaseModel):
    qid: str
    query: str
    passage: str
    original_index: int

class PairRerankRequest(BaseModel):
    pairs: List[Pair] = Field(..., description="(qid, query, passage) tuples")
    top_k_per_q: int = Field(default=3, ge=1, le=100)

class PairTopResult(BaseModel):
    passage: str
    score: float
    original_index: int

class PairRerankResponse(BaseModel):
    results: Dict[str, List[PairTopResult]]  # qid -> top_k list
    timing: Dict[str, float]
    total_pairs: int

@dataclass
class ProcessingJob:
    job_id: str
    query: str
    passages: List[str]
    top_k: int
    future: asyncio.Future
    timestamp: float

class AsyncRerankerEngine:
    """High-performance async reranker engine optimized for A100 GPU"""

    def __init__(self):
        self.model = None
        self.tokenizer = None
        self.device = None
        self.processing_queue = asyncio.Queue(maxsize=1000)
        self.batch_processor_task = None
        self.stats = {
            "requests_processed": 0,
            "total_processing_time": 0,
            "avg_batch_size": 0,
            "gpu_utilization": 0
        }
        self.lock = asyncio.Lock()

    async def initialize(self):
        """Initialize the reranker model with optimal settings for A100"""
        print("🚀 Initializing BGE-reranker-v2-m3 for A100 GPU...")
        torch.set_float32_matmul_precision("high")
        start_time = time.time()

        # Setup device with optimal settings
        if torch.cuda.is_available():
            self.device = torch.device('cuda')
            # Enable optimizations for A100
            torch.backends.cudnn.benchmark = True
            torch.backends.cuda.matmul.allow_tf32 = True
            torch.backends.cudnn.allow_tf32 = True

            # Set memory allocation strategy
            torch.cuda.empty_cache()
            torch.cuda.set_per_process_memory_fraction(0.9)  # Use 90% of GPU memory

            print(f"🎯 GPU: {torch.cuda.get_device_name(0)}")
            print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB")
        else:
            self.device = torch.device('cpu')
            print("⚠️  Using CPU - GPU not available")

        # Load model with optimizations
        self.model = FlagReranker(
            MODEL_NAME,
            use_fp16=True,  # Use half precision for speed
            device=self.device
        )

        # Warmup with different batch sizes to optimize memory
        print("🔥 Warming up model with various batch sizes...")
        warmup_queries = ["test query"] * 16
        warmup_passages = ["test passage"] * 16
        warmup_pairs = [[q, p] for q, p in zip(warmup_queries, warmup_passages)]

        # Warmup with different batch sizes
        for batch_size in [1, 4, 8, 16, 32]:
            if batch_size <= len(warmup_pairs):
                _ = self.model.compute_score(warmup_pairs[:batch_size])

        # Clear cache after warmup
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        print(f"✅ Model initialized in {time.time() - start_time:.2f}s")

        # Start background batch processor
        self.batch_processor_task = asyncio.create_task(self._batch_processor())

    async def _batch_processor(self):
        """Background task that processes queued requests in optimized batches"""
        while True:
            try:
                # Collect jobs for batching
                jobs = []
                deadline = time.time() + 0.025  # 10ms collection window

                # Collect initial job
                try:
                    job = await asyncio.wait_for(self.processing_queue.get(), timeout=1.0)
                    jobs.append(job)
                except asyncio.TimeoutError:
                    continue

                # Collect additional jobs within time window
                while len(jobs) < BATCH_SIZE and time.time() < deadline:
                    try:
                        job = await asyncio.wait_for(self.processing_queue.get(), timeout=0.001)
                        jobs.append(job)
                    except asyncio.TimeoutError:
                        break

                if jobs:
                    await self._process_job_batch(jobs)

            except Exception as e:
                print(f"❌ Error in batch processor: {e}")
                await asyncio.sleep(0.1)



    async def _process_job_batch(self, jobs: List[ProcessingJob]):
      batch_start = time.time()
      try:
        all_pairs = []
        job_slices = []
        for job in jobs:
            start = len(all_pairs)
            all_pairs.extend([[job.query, p] for p in job.passages])
            end = len(all_pairs)
            job_slices.append((job, start, end))

        # ONE compute (internally batched by batch_size)
        scores = await asyncio.get_event_loop().run_in_executor(
            None, self._compute_scores_sync, all_pairs
        )

        # fan-out
        cursor = 0
        for (job, start, end) in job_slices:
            job_scores = scores[start:end]
            ranked = [{
                "passage": passage,
                "score": float(score),
                "original_index": idx
            } for idx, (passage, score) in enumerate(zip(job.passages, job_scores))]
            ranked.sort(key=lambda x: x["score"], reverse=True)
            top_results = ranked[:job.top_k]
            if not job.future.done():
                job.future.set_result(top_results)

        batch_time = time.time() - batch_start
        async with self.lock:
            self.stats["requests_processed"] += len(jobs)
            self.stats["total_processing_time"] += batch_time
            self.stats["avg_batch_size"] = (self.stats["avg_batch_size"] + len(jobs)) / 2
        print(f"⚡ Processed batch of {len(jobs)} jobs in {batch_time:.3f}s")

      except Exception as e:
        print(f"❌ Error processing job batch: {e}")
        for job in jobs:
            if not job.future.done():
                job.future.set_exception(e)

    def _compute_scores_sync(self, pairs: List[List[str]]) -> List[float]:
        """Synchronous score computation for thread execution"""
        with torch.no_grad():
            scores = self.model.compute_score(pairs, batch_size=min(BATCH_SIZE, len(pairs)))
            if isinstance(scores, float):
                return [scores]
            return scores.tolist() if hasattr(scores, 'tolist') else list(scores)

    async def rerank_async(self, query: str, passages: List[str], top_k: int) -> List[Dict]:
        """Async rerank operation"""
        if not passages:
            return []

        # Create job
        job_id = str(uuid.uuid4())
        future = asyncio.Future()
        job = ProcessingJob(
            job_id=job_id,
            query=query,
            passages=passages,
            top_k=min(top_k, len(passages)),
            future=future,
            timestamp=time.time()
        )

        # Queue job
        try:
            await asyncio.wait_for(self.processing_queue.put(job), timeout=1.0)
        except asyncio.TimeoutError:
            raise HTTPException(status_code=503, detail="Reranker queue is full")

        # Wait for result
        try:
            result = await asyncio.wait_for(future, timeout=PROCESSING_TIMEOUT)
            return result
        except asyncio.TimeoutError:
            raise HTTPException(status_code=504, detail="Reranking timeout")

    async def get_stats(self) -> Dict:
        """Get performance statistics"""
        async with self.lock:
            stats = self.stats.copy()
            if stats["requests_processed"] > 0:
                stats["avg_processing_time"] = stats["total_processing_time"] / stats["requests_processed"]
            else:
                stats["avg_processing_time"] = 0
            return stats

    async def shutdown(self):
        """Clean shutdown"""
        if self.batch_processor_task:
            self.batch_processor_task.cancel()
            try:
                await self.batch_processor_task
            except asyncio.CancelledError:
                pass


In [5]:
# Global reranker instance
reranker_engine = AsyncRerankerEngine()

In [6]:
# FastAPI app
app = FastAPI(
    title="High-Performance Async Reranker",
    description="Ultra-fast parallel reranking service optimized for A100 GPU",
    version="2.0.0"
)

# Add CORS middleware
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.on_event("startup")
async def startup_event():
    """Initialize reranker on startup"""
    await reranker_engine.initialize()

@app.on_event("shutdown")
async def shutdown_event():
    """Clean shutdown"""
    await reranker_engine.shutdown()

@app.get("/health")
async def health_check():
    """Health check endpoint"""
    stats = await reranker_engine.get_stats()
    return {
        "status": "healthy",
        "model": MODEL_NAME,
        "device": str(reranker_engine.device),
        "gpu_available": torch.cuda.is_available(),
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A",
        "queue_size": reranker_engine.processing_queue.qsize(),
        "stats": stats
    }

@app.post("/rerank", response_model=RerankResponse)
async def rerank(request: RerankRequest):
    """Single query reranking endpoint"""
    start_time = time.time()

    if not request.query or not request.passages:
        raise HTTPException(status_code=400, detail="Query and passages are required")

    try:
        # Perform reranking
        top_results = await reranker_engine.rerank_async(
            request.query,
            request.passages,
            request.top_k
        )

        total_time = time.time() - start_time

        return RerankResponse(
            query=request.query,
            top_passages=top_results,
            timing={
                "total_time": round(total_time, 3),
                "queue_wait_time": 0.0  # Could be enhanced to track this
            },
            total_passages=len(request.passages),
            top_k=min(request.top_k, len(request.passages)),
            request_id=request.request_id
        )

    except Exception as e:
        print(f"❌ Error in rerank endpoint: {e}")
        raise HTTPException(status_code=500, detail=str(e))



@app.post("/rerank_pairs", response_model=PairRerankResponse)
async def rerank_pairs(request: PairRerankRequest):
    if not request.pairs:
        raise HTTPException(status_code=400, detail="pairs cannot be empty")
    start = time.time()

    # One big compute (best perf). We bypass the queue on purpose.
    pairs = [[p.query, p.passage] for p in request.pairs]

    # Use the engine's optimized sync scorer in a thread
    scores = await asyncio.get_event_loop().run_in_executor(
        None, reranker_engine._compute_scores_sync, pairs
    )

    # Group back by qid and select top_k_per_q
    by_qid: Dict[str, List[Tuple[int, float]]] = {}
    for i, p in enumerate(request.pairs):
        by_qid.setdefault(p.qid, []).append((i, scores[i]))

    out: Dict[str, List[PairTopResult]] = {}
    for qid, indices_scores in by_qid.items():
        # sort desc by score
        indices_scores.sort(key=lambda x: x[1], reverse=True)
        chosen = []
        for idx, sc in indices_scores[:request.top_k_per_q]:
            pp = request.pairs[idx]
            chosen.append(PairTopResult(
                passage=pp.passage,
                score=float(sc),
                original_index=pp.original_index
            ))
        out[qid] = chosen

    return PairRerankResponse(
        results=out,
        timing={"total_time": round(time.time() - start, 3)},
        total_pairs=len(request.pairs)
    )

@app.post("/batch_rerank", response_model=BatchRerankResponse)
async def batch_rerank(request: BatchRerankRequest):
    start_time = time.time()
    if not request.queries or not request.passages:
        raise HTTPException(status_code=400, detail="Queries and passages are required")
    try:
        tasks = [
            asyncio.create_task(
                reranker_engine.rerank_async(q, request.passages, request.top_k)
            )
            for q in request.queries
        ]
        results_raw = await asyncio.gather(*tasks)

        results = []
        for q, top_results in zip(request.queries, results_raw):
            results.append(RerankResponse(
                query=q,
                top_passages=top_results,
                timing={"processing_time": 0.0},
                total_passages=len(request.passages),
                top_k=min(request.top_k, len(request.passages))
            ))

        total_time = time.time() - start_time
        return BatchRerankResponse(
            results=results,
            timing={
                "total_time": round(total_time, 3),
                "avg_time_per_query": round(total_time / len(request.queries), 3),
                "parallel_speedup": f"{len(request.queries)}x"
            },
            request_id=request.request_id
        )
    except Exception as e:
        print(f"❌ Error in batch_rerank endpoint: {e}")
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/stats")
async def get_performance_stats():
    """Get detailed performance statistics"""
    stats = await reranker_engine.get_stats()

    # Add GPU memory info if available
    if torch.cuda.is_available():
        stats["gpu_memory"] = {
            "allocated": torch.cuda.memory_allocated() / 1e9,
            "cached": torch.cuda.memory_reserved() / 1e9,
            "max_allocated": torch.cuda.max_memory_allocated() / 1e9
        }

    return stats


/tmp/ipython-input-2666560312.py:17: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("startup")
/tmp/ipython-input-2666560312.py:22: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("shutdown")


In [7]:
def start_ngrok():
    """Start ngrok tunnel"""
    print("🌐 Starting ngrok tunnel...")

    # Create tunnel
    public_url = ngrok.connect(8000)  # FastAPI default port
    print(f"✅ Ngrok tunnel established: {public_url}")
    print(f"📋 Reranker API URL: {public_url}/rerank")
    print(f"📋 Batch API URL: {public_url}/batch_rerank")
    print(f"📋 Health check URL: {public_url}/health")
    print(f"📋 Stats URL: {public_url}/stats")
    print(f"📋 Interactive docs: {public_url}/docs")

    return public_url

async def start_server():
    """Start the FastAPI server in Colab environment"""
    import uvicorn

    # Start ngrok tunnel
    public_url = start_ngrok()

    # Save the URL for easy access
    with open('/content/reranker_url.txt', 'w') as f:
        f.write(str(public_url))

    print("\n🚀 High-Performance Async Reranker is ready!")
    print(f"Use this URL in your RAG app: {public_url}/rerank")
    print("="*60)

    # Create uvicorn server config for Colab
    config = uvicorn.Config(
        app,
        host="0.0.0.0",
        port=8000,
        access_log=False,
        log_level="info"
    )

    server = uvicorn.Server(config)
    await server.serve()


In [8]:
from pyngrok import ngrok
from google.colab import userdata



# Set the authtoken
ngrok.set_auth_token("315EiZgtm80SnsDsWv9sD4Es85e_ifgKnxoeQ7jVomqXehZE")



In [ ]:
if __name__ == "__main__":
    # Run in existing event loop (Colab compatible)
    import nest_asyncio
    nest_asyncio.apply()

    asyncio.run(start_server())

🌐 Starting ngrok tunnel...
✅ Ngrok tunnel established: NgrokTunnel: "https://bb5e5fbf2d8c.ngrok-free.app" -> "http://localhost:8000"
📋 Reranker API URL: NgrokTunnel: "https://bb5e5fbf2d8c.ngrok-free.app" -> "http://localhost:8000"/rerank
📋 Batch API URL: NgrokTunnel: "https://bb5e5fbf2d8c.ngrok-free.app" -> "http://localhost:8000"/batch_rerank
📋 Health check URL: NgrokTunnel: "https://bb5e5fbf2d8c.ngrok-free.app" -> "http://localhost:8000"/health
📋 Stats URL: NgrokTunnel: "https://bb5e5fbf2d8c.ngrok-free.app" -> "http://localhost:8000"/stats
📋 Interactive docs: NgrokTunnel: "https://bb5e5fbf2d8c.ngrok-free.app" -> "http://localhost:8000"/docs

🚀 High-Performance Async Reranker is ready!
Use this URL in your RAG app: NgrokTunnel: "https://bb5e5fbf2d8c.ngrok-free.app" -> "http://localhost:8000"/rerank


INFO:     Started server process [1008]
INFO:     Waiting for application startup.


🚀 Initializing BGE-reranker-v2-m3 for A100 GPU...
🎯 GPU: NVIDIA A100-SXM4-40GB
💾 GPU Memory: 42.5GB


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

🔥 Warming up model with various batch sizes...


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


✅ Model initialized in 16.58s


Compute Scores: 100%|██████████| 1/1 [00:01<00:00,  1.09s/it]
